# __Lecture 09: Practical analyses in Python__

----
### **_Be sure to pull the latest notebook!!_**

#### __Announcement:__ Homework 4 is assigned today and is due November 5.

#### __On Tuesday (lecture 08), we:__
- motivated pattern recognition
- read and dissected regular expressions with the `re` module
- practiced writing patterns of our own

#### __Today we will:__
- warm up with a few regular expression predictions
- use the `biopython` module to read a real FASTQ file and work with sequences
- write __test cases__ to check that our code does what we think it does
- spend the second half of lecture on __one long interactive example__ that combines the `re` and
  `biopython` skills above: parsing barcodes out of a real sequencing run

__Homework 4 extends today's analysis__, so keep this notebook handy.

###  
----

## __Review:__ regular expression predictions

The pattern cheat sheet is in [lecture08.ipynb](../lecture08/lecture08.ipynb) - keep it open.

For each cell below, __predict the output__, then run it. 

In [ ]:
import re

# 1. Which part of the strain name does each pattern pull out?
strain = "A/Puerto Rico/8/1934 (H1N1)"

print(re.compile(r"\d{4}").search(strain))
print(re.compile(r"(?P<city>[A-Za-z ]+)/\d").search(strain).group("city"))

In [ ]:
# 2. This pattern is SUPPOSED to pull a 6-nucleotide barcode out of a read.
#    What does it return for each of these two reads? Is each answer correct?
barcode_pattern = re.compile(r"CTAGC(?P<barcode>[A-Z]{6})GATCA")

for read in ["CTAGCATCGATGATCA", "CTAGCATYGATGATCA"]:
    match = barcode_pattern.search(read)
    print(read, "->", match.group("barcode") if match else "no match")

The second read contains a `Y`, which is not a nucleotide - but the pattern matches it anyway
and happily hands back a barcode. A "wrong" pattern fails silently, and we have to carefully read the outputs to detect the failure. We will do something better than that today.

----
# __New module: `biopython`__

The [Biopython](https://biopython.org/) package is a long-standing collection of tools for
computational biology. We will use two small pieces of it:

- [`Bio.SeqIO`](https://biopython.org/wiki/SeqIO) - reading and writing sequence files (FASTA, FASTQ,
  GenBank, and ~30 other formats). 
- [`Bio.Seq`](https://biopython.org/wiki/Seq) - a sequence object that knows it is a biological
  sequence, with methods like `reverse_complement()`, `transcribe()`, and `translate()`.

#### The FASTQ format

Our data file, `barcodes_R1.fastq`, is in FASTQ format: __four lines per read__.

    @HISEQ:621:HMJGNBCX2:1:1101:1797:2150 1:N:0:ATGTCA   <- read name (the Illumina header)
    GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATT...           <- the sequence
    +                                                     <- a separator
    GGGGGGIIIGIIIIGIIIIGIIIIIIIGGIGIIIIIIII...           <- one quality character per base

In [ ]:
import Bio.SeqIO
import Bio.Seq

# SeqIO.parse returns reads one at a time; list() pulls them all into memory
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

print(f"Found {len(seqreads)} sequencing reads.")

#### Reads come back as `SeqRecord` objects

A [SeqRecord](https://biopython.org/wiki/SeqRecord) bundles together everything the file knew
about one read: its name, its sequence, and its per-base quality scores.

In [ ]:
read = seqreads[0]

print(type(read))
print(read)

In [ ]:
# the pieces you will actually use:
print("id:      ", read.id)
print("sequence:", read.seq)
print("length:  ", len(read.seq))
print("quality: ", read.letter_annotations["phred_quality"])

#### Sequences are `Seq` objects, not strings

A `Seq` behaves a lot like a string (you can index it, slice it, and take its length), but it also
has biology-aware methods. If you need to hand the sequence to something that expects a plain
string - including the `re` module - convert it with `str()`.

In [ ]:
seq = seqreads[0].seq

print(type(seq))
print(seq[:30])

# converting to a string when we need one
seq_string = str(seq)
print(type(seq_string))

In [ ]:
# a few of the built-in Seq methods
short = Bio.Seq.Seq("ATGGCCATTGTAATGGGCCGC")

print("sequence:          ", short)
print("complement:        ", short.complement())
print("reverse complement:", short.reverse_complement())
print("transcribed (RNA): ", short.transcribe())
print("translated:        ", short.translate())

__Note how much work `reverse_complement()` is doing for you__ - compare it with the hand-written
`reverse_complement` function from lecture 6. 

#### Another note: `biopython` is a great default for reading sequence files and for small-to-medium analyses like this one. It is not always the fastest option. 

----
# __A real analysis: parsing barcodes from a sequencing run__

__What this section is:__ one long worked example that combines Tuesday's `re` skills and today's `biopython` skills on a real dataset, and adding one new habit (writing test cases).
We will spend the rest of lecture in this section.

The reads in `barcodes_R1.fastq` come from a real sequencing run of influenza virus HA and NA
genes. Each molecule in the library was built like this:

    5'-[ end of HA or NA ]-AGGCGGCCGC-[ 16-nucleotide barcode ]-3'

The sequencer read these molecules __from the other end__, so what is actually in our file is the
reverse complement:

    5'-[ reverse complement of the barcode ]-GCGGCCGCCT-[ reverse complement of the end of HA/NA ]-3'

You can see this in the very first read: the first 16 bases are the (reverse complemented) barcode,
immediately followed by `GCGGCCGCCT`.

In [ ]:
print(str(seqreads[0].seq)[:40])

#### In this analysis, we want to parse these reads and extract the barcodes using regular expressions. We will build it in numbered steps, starting here with step 0.

### __Step 0: say what "correct" means, before writing the pattern__

Before writing any code, we write the requirements down, in plain English. After reverse
complementing, a valid read should satisfy all of:

__Requirements:__
1. the read contains the fixed upstream sequence `AGGCGGCCGC`
2. the barcode is exactly __16__ bases long
3. every barcode base is one of `A`, `C`, `G`, `T`, `N` - and nothing else
4. the barcode is reported in upper case, in the forward orientation

Anything that fails one of these is not a valid barcode, and the parser function that you will write in step 3 should return `None` for it.


### __Step 1: turn the requirements into test cases__

Those requirements are only useful if we actually check them with tests.

A __test case__ is one input paired with the answer you expect. You have already seen the idea when we used `assert` statements on Tuesday. A test case is the same thing, except that we collect them in a list and run the whole set at once, which
is how real test suites work.

Each requirement usually needs two kinds of case:

* a __positive__ case - a good read, and the barcode it should give back
* a __negative__ case - an input that violates __that one requirement__, for which the parser
  should return `None`

#### Wait - what are we testing?

The parser does not exist yet. That is deliberate, and we already know enough to test it, because
the requirements above describe it. The function will be called __`read_barcode`__, and:

* __it takes__ one read, in the orientation of the __file__ (ending with `GCGGCCGCCT`)
* __it returns__ the barcode as an upper-case string, in the forward orientation
* __or it returns `None`__, if the read does not meet the requirements

That is all you need in order to write down inputs and the answers you expect.

#### How to format a test case

Today, each test case is a __tuple of three things__:

    (read, expected result, label)

* __read__ - the input sequence, as a string, in the orientation of the __file__ (so it ends with
  `GCGGCCGCCT`; it does not start with `AGGCGGCCGC`)
* __expected result__ - the barcode string you expect back for a good read, or `None` if this read
  should be rejected
* __label__ - a short string saying which requirement this case is checking, so that a failure
  message tells you what broke

The whole set of cases goes in one list, named `test_cases`.

__Your task:__ add a test case for each of requirements 1-4 to the list below. One positive case and
one negative case are already there to show the format. Nothing to run yet - you'll write the parser function called `read_barcode` and test it with these test cases in step 3.

In [ ]:
test_cases = [
    # (read, expected result, label saying which requirement this checks)
    ("TTTTTTTTTTTTTTTTGCGGCCGCCT", "AAAAAAAAAAAAAAAA", "positive: clean read"),
    ("TTTTTTTACGTTTTTTGCGGGCCT", None, "requirement 1: upstream sequence is wrong"),
    # YOUR TEST CASES HERE
]

### __Step 2: a helper for running the whole list - `run_tests`__

This function will allow you to run a list of test cases for your parser function that you will write in the next step.

It takes two arguments:

* `parser_function` - the function you want to test (for us, `read_barcode`)
* `test_cases` - the list of `(read, expected result, label)` tuples you just wrote

and it prints `PASS` or `FAIL` for each case, plus a count of the failures at the end. We will call
it in step 3, once there is a function to test.

In [ ]:
# a helper I am giving you: it runs a parser function against a list of test cases
# and reports the results
def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

### __Step 3: write the parsing function, and use your tests to check it__

The documentation is written for you. __Write the body of this function__, then run
`run_tests(read_barcode, test_cases)` in the cell below it and keep fixing the function until all
your cases pass.

_Hint 1: `seqread.reverse_complement()` does the orientation flip for you._

_Hint 2: `re` works on strings, not `Seq` objects - convert with `str()` first._

_Hint 3: use the input variables `upstream` and `bclen` to build your pattern._

In [ ]:
def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.

    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'AAAAAAAAAAAAAAAA'
    """
    # your code here...

    return None

In [ ]:
run_tests(read_barcode, test_cases)

### __Step 4: run it on all 10,000 reads__

Now the part that is only possible because we trust the function: apply it to the whole file and
__count the unique barcodes__. Keep track of the reads that __did not__ match as well
an error.

In [ ]:
barcode_counts = {}
unmatched = []

for read in seqreads:
    # your code here

----
# __Wrapping up__

#### The workflow from these two lectures

1. __Describe the design in plain English__ before writing any code
2. __Write test cases__ that encode that description, including negatives
3. __Write the pattern__, and run it against the test cases
4. __Run analysis on real data__

#### Where this goes next

- __Homework 4__ extends today's analysis: the reads come from __either__ HA or NA, and you will
  work out which, using the sequence at the far end of the read. Almost all of the code you need is
  in this notebook.
- __Lecture 15__ (Introduction to sequencing data) picks up the sequencing side in much more depth.
- __Lecture 17__ (Immune repertoire sequencing) applies this same kind of parsing to a different
  biological problem.